# Proyecto de regresión lineal - Medical Insurance Cost

En este notebook voy a resolver el ejercicio de regresión lineal con el dataset de costes médicos.  
La idea es hacerlo paso a paso, viendo un poco los datos, limpiándolos, preparando las variables y entrenando un modelo sencillo para predecir `charges`.


## 1. Importar librerías

Primero importo las librerías que voy a usar.  
Las dejo aquí juntas para tener el notebook más ordenado desde el principio.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)


## 2. Cargar los datos

Cargo el dataset desde la URL del ejercicio para trabajar directamente sobre él.


In [ ]:
url = "https://raw.githubusercontent.com/4GeeksAcademy/linear-regression-project-tutorial/main/medical_insurance_cost.csv"
total_data = pd.read_csv(url)
total_data.head()


## 3. Revisión rápida

Antes de entrenar nada, reviso el tamaño del dataset, los tipos de datos, los nulos y si hay duplicados.


In [ ]:
total_data.shape

In [ ]:
total_data.info()

In [ ]:
total_data.isnull().sum()

In [ ]:
total_data.duplicated().sum()

## 4. Limpieza inicial

En este caso elimino los duplicados para quedarme con una versión limpia del dataset.


In [ ]:
total_data = total_data.drop_duplicates().reset_index(drop=True)
total_data.shape


## 5. EDA rápido

Aquí hago una revisión visual sencilla para entender mejor cómo se comportan algunas variables.


In [ ]:
total_data.describe().T

In [ ]:
fig, axis = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(data=total_data, x="age", kde=True, ax=axis[0, 0])
axis[0, 0].set_title("Distribución de age")

sns.histplot(data=total_data, x="bmi", kde=True, ax=axis[0, 1])
axis[0, 1].set_title("Distribución de bmi")

sns.histplot(data=total_data, x="children", kde=False, ax=axis[1, 0])
axis[1, 0].set_title("Distribución de children")

sns.histplot(data=total_data, x="charges", kde=True, ax=axis[1, 1])
axis[1, 1].set_title("Distribución de charges")

plt.tight_layout()
plt.show()


In [ ]:
fig, axis = plt.subplots(1, 3, figsize=(15, 4))

sns.boxplot(data=total_data, y="charges", ax=axis[0])
axis[0].set_title("Boxplot de charges")

sns.boxplot(data=total_data, x="smoker", y="charges", ax=axis[1])
axis[1].set_title("Charges según smoker")

sns.boxplot(data=total_data, x="sex", y="charges", ax=axis[2])
axis[2].set_title("Charges según sex")

plt.tight_layout()
plt.show()


In [ ]:
fig, axis = plt.subplots(1, 3, figsize=(16, 4))

sns.countplot(data=total_data, x="sex", ax=axis[0])
axis[0].set_title("Conteo por sex")

sns.countplot(data=total_data, x="smoker", ax=axis[1])
axis[1].set_title("Conteo por smoker")

sns.countplot(data=total_data, x="region", ax=axis[2])
axis[2].set_title("Conteo por region")

plt.tight_layout()
plt.show()


## 6. Pasar texto a números y escalar

Como la regresión lineal necesita variables numéricas, convierto las columnas categóricas a números.  
Después escalo las variables con Min-Max para dejarlas en una escala parecida.


In [ ]:
total_data["sex_n"] = pd.factorize(total_data["sex"])[0]
total_data["smoker_n"] = pd.factorize(total_data["smoker"])[0]
total_data["region_n"] = pd.factorize(total_data["region"])[0]

num_variables = ["age", "bmi", "children", "sex_n", "smoker_n", "region_n"]

scaler = MinMaxScaler()
scal_features = scaler.fit_transform(total_data[num_variables])

total_data_scal = pd.DataFrame(scal_features, index=total_data.index, columns=num_variables)
total_data_scal["charges"] = total_data["charges"]

total_data_scal.head()


## 7. Relación con la variable objetivo

Aquí miro un poco qué variables parecen tener más relación con `charges`.


In [ ]:
fig, axis = plt.subplots(3, 2, figsize=(12, 14))

sns.regplot(data=total_data_scal, x="age", y="charges", ax=axis[0, 0], line_kws={"color": "red"})
axis[0, 0].set_title("age vs charges")

sns.regplot(data=total_data_scal, x="bmi", y="charges", ax=axis[0, 1], line_kws={"color": "red"})
axis[0, 1].set_title("bmi vs charges")

sns.regplot(data=total_data_scal, x="children", y="charges", ax=axis[1, 0], line_kws={"color": "red"})
axis[1, 0].set_title("children vs charges")

sns.regplot(data=total_data_scal, x="sex_n", y="charges", ax=axis[1, 1], line_kws={"color": "red"})
axis[1, 1].set_title("sex_n vs charges")

sns.regplot(data=total_data_scal, x="smoker_n", y="charges", ax=axis[2, 0], line_kws={"color": "red"})
axis[2, 0].set_title("smoker_n vs charges")

sns.regplot(data=total_data_scal, x="region_n", y="charges", ax=axis[2, 1], line_kws={"color": "red"})
axis[2, 1].set_title("region_n vs charges")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(total_data_scal.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Heatmap de correlación")
plt.show()


## 8. Selección de variables

Ahora separo X e y, hago train/test y me quedo con las variables que más relación tienen con la variable objetivo usando `SelectKBest`.


In [ ]:
X = total_data_scal.drop("charges", axis=1)
y = total_data_scal["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

selection_model = SelectKBest(score_func=f_regression, k=4)
selection_model.fit(X_train, y_train)

selected_columns = X_train.columns[selection_model.get_support()]
selected_columns


In [ ]:
X_train_sel = pd.DataFrame(selection_model.transform(X_train), columns=selected_columns)
X_test_sel = pd.DataFrame(selection_model.transform(X_test), columns=selected_columns)

X_train_sel.head()


## 9. Guardar datos procesados

Guardo una copia del train y del test por si luego quiero reutilizarlos sin repetir todo el preprocesado.


In [ ]:
X_train_sel["charges"] = y_train.values
X_test_sel["charges"] = y_test.values

os.makedirs("data/processed", exist_ok=True)

X_train_sel.to_csv("data/processed/clean_train.csv", index=False)
X_test_sel.to_csv("data/processed/clean_test.csv", index=False)


In [ ]:
train_data = pd.read_csv("data/processed/clean_train.csv")
test_data = pd.read_csv("data/processed/clean_test.csv")

train_data.head()


## 10. Modelo de regresión lineal

Ahora entreno una regresión lineal básica con las variables seleccionadas.


In [ ]:
X_train = train_data.drop("charges", axis=1)
y_train = train_data["charges"]

X_test = test_data.drop("charges", axis=1)
y_test = test_data["charges"]

model = LinearRegression()
model.fit(X_train, y_train)


In [ ]:
print(f"Intercept: {model.intercept_}")
print(f"Coeficientes: {model.coef_}")


## 11. Predicciones y evaluación

Hago las predicciones sobre test y reviso las métricas principales.


In [ ]:
y_pred = model.predict(X_test)
y_pred[:10]


In [ ]:
print(f"MSE: {mean_squared_error(y_test, y_pred)}")
print(f"R2 Score: {r2_score(y_test, y_pred)}")


In [ ]:
results = pd.DataFrame({
    "Real": y_test.values,
    "Predicción": y_pred
})

results.head(10)


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Valores reales")
plt.ylabel("Predicciones")
plt.title("Valores reales vs predicciones")
plt.show()


## 12. Guardar el modelo

Por último guardo el modelo por si luego quiero reutilizarlo sin volver a entrenarlo.


In [ ]:
from pickle import dump

os.makedirs("models", exist_ok=True)
dump(model, open("models/linear_regression_medical_insurance.sav", "wb"))


## Conclusión

En este proyecto he cargado el dataset, he revisado los datos, he eliminado duplicados, he transformado las variables categóricas a numéricas, he escalado los datos, he seleccionado las variables más útiles y he entrenado un modelo de regresión lineal.

No es un análisis súper complejo, pero sí me sirve para entender bien el flujo completo de un problema de regresión y para montar un modelo sencillo y funcional.
